# 🌧️ 2권 · 타깃과 강우 — "비가 오면 물이 얼마나 오르나?"

> 시리즈: 1권 구조·품질 → **2권 타깃과 강우** → 3권 수위 변화·시계열 → 4권 관측소 유형·유역환경 → 5권 연결망 → 6권 검증 전략·오차 위치 · 별책 조석

### 한 줄 요약
**타깃은 대부분 0 근처인데 가끔 몇 m씩 튀고, 그 몇 %가 점수를 결정해요. 비가 많을수록 타깃의 평균도 커지지만 '얼마나 커질지 모르는 정도(분산)'도 같이 커져요. 비 신호는 관측소 거의 전부에서 같은 방향이에요.**

| 장 | 질문 |
|---|---|
| 1 | 타깃은 어떻게 생겼나 — 0 근처에 몰리고 꼬리가 길다 |
| 2 | 강우 세 가지(위성·지상·레이더)는 어떻게 생겼고 서로 얼마나 맞나 |
| 3 | 비가 오면 타깃은 어떻게 변하나 — 평균과 분산이 같이 큰다 |
| 4 | 이 관계가 관측소 전부에서 성립하나 |
| 5 | 변수별 상관 요약 — 관계는 단조롭지만 곧은 직선은 아니다 |
| 6 | 누수 점검과 정리 |

### 용어 사전

| 용어 | 뜻 | 예시 |
|---|---|---|
| **IMERG** | 위성으로 추정한 강우(mm), 약 10km 격자 | 구름 위에서 본 비 |
| **지상강수** (`sfc_rain`) | 우량계로 잰 강우(0.5mm 계단, 1권) | 땅에서 받은 비 |
| **레이더 반사도** | 레이더 전파가 빗방울에 맞고 돌아온 세기. 강우량이 아니라 "비구름이 얼마나 짙은가" | 일기예보 레이더 영상의 색 |
| **이분산** | 어떤 조건에서 값의 흩어짐(분산)이 달라지는 것 | 비가 많을수록 타깃이 들쭉날쭉 |
| **Pearson / Spearman** | Pearson은 직선 관계, Spearman은 "순서가 같이 가나"(단조 관계)를 재는 상관 | 곡선이라도 계속 오르면 Spearman은 높다 |
| **비조석** | 조석 15곳과 고장 관측소 1곳을 뺀 나머지(별책 참고) | |

## 0. 준비

그림 설정, 색 약속, 특수 관측소 목록(조석 등), 도우미 함수를 불러와요. 색 약속: ⚪ 회색 = 전체 · 🔵 파랑 = 비조석(일반) · 🟠 주황 = 조석/강조 · 🟡 노랑 = 강우

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.rcParams["font.family"] = ["DejaVu Sans", "AppleGothic"]      # 숫자는 DejaVu, 한글은 AppleGothic
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True, "legend.frameon": False})

EDA = Path.cwd() if (Path.cwd() / "cache").exists() else Path.cwd().parent
CACHE, RESULTS = EDA / "cache", EDA / "cache" / "results"
TARGET = "target_maxrise_6h"
WL = ["wl_t", "wl_t_minus_1h", "wl_t_minus_3h", "wl_t_minus_6h", "wl_t_minus_12h", "wl_t_minus_24h"]
DWL = ["dwl_1h", "dwl_3h", "dwl_6h", "dwl_12h", "dwl_24h"]
IMERG = ["imerg_rain_1h", "imerg_rain_6h", "imerg_rain_24h"]
SFC = ["sfc_rain_1h", "sfc_rain_6h", "sfc_rain_24h"]
RADAR = ["radar_reflectivity_mean", "radar_reflectivity_p95", "radar_reflectivity_ge20_fraction"]
ENV = ["cat_area", "wsarea", "mean_elevation", "mean_slope", "urban_ratio", "forest_ratio", "stream_order"]
FEATURES = WL + IMERG + SFC + RADAR + ENV

# 별책 tidal_analysis.ipynb에서 판정한 특수 관측소 (dwl_1h 자기상관 기준, 타깃 미사용)
TIDAL = ["ST0358", "ST0292", "ST0326", "ST0347", "ST0159", "ST0110", "ST0313", "ST0415",
         "ST0234", "ST0129", "ST0192", "ST0051", "ST0130", "ST0164", "ST0123"]
WEAK_TIDAL = ["ST0151", "ST0193", "ST0226", "ST0229", "ST0260"]
OPERATED_24H = ["ST0194", "ST0200"]
STUCK = ["ST0027"]
SPECIAL = TIDAL + WEAK_TIDAL + OPERATED_24H + STUCK

# 색 약속: 회색 = 전체, 파랑 = 비조석(일반), 주황 = 조석/강조, 노랑 = 강우
C_ALL, C_NT, C_TIDAL, C_RAIN, C_GREEN, C_PINK, C_PURPLE = "#9a9890", "#2a78d6", "#eb6834", "#eda100", "#1baf7a", "#e87ba4", "#4a3aa7"
CAT = [C_NT, C_TIDAL, C_GREEN, C_RAIN, C_PINK, "#008300", C_PURPLE, "#e34948"]


def keyed(d):
    """(관측소, 시각) → 정수 키 하나. 예: 관측소 12, 시각 345 → 12_000_345"""
    return d["station_code"].to_numpy(np.int64) * 1_000_000 + d["hour"].to_numpy(np.int64)


def shift_lookup(d, key, col, dh):
    """같은 관측소의 (지금 + dh)시간 col 값. 그 행이 없으면 NaN. dh < 0 이면 과거, dh > 0 이면 미래."""
    return pd.Series(d[col].to_numpy(np.float64), index=key).reindex(key + dh).to_numpy()


def station_corr(d, x, y, min_n=50, by="station_id"):
    """그룹(기본: 관측소)별 피어슨 상관을 합계 통계로 한 번에 계산."""
    t = pd.DataFrame({"s": d[by].to_numpy(), "x": x, "y": y}).dropna()
    t["xy"], t["xx"], t["yy"] = t.x * t.y, t.x * t.x, t.y * t.y
    g = t.groupby("s").agg(n=("x", "size"), sx=("x", "sum"), sy=("y", "sum"), sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = g.sxy / g.n - (g.sx / g.n) * (g.sy / g.n)
    return (cov / np.sqrt((g.sxx / g.n - (g.sx / g.n) ** 2) * (g.syy / g.n - (g.sy / g.n) ** 2))).where(g.n > min_n)


def symlog_x(ax, linthresh=0.1):
    """0 근처는 선형, 멀리는 로그인 x축"""
    ax.set_xscale("symlog", linthresh=linthresh)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))


def binned(x, y, q=20):
    """x를 구간으로 나눠 y의 평균·표준편차. x가 0인 행은 따로 한 칸."""
    x, y = pd.Series(np.asarray(x, float)), pd.Series(np.asarray(y, float))
    k = pd.Series("0", index=x.index, dtype=object)
    nz = x != 0
    if nz.sum() > q:
        k[nz] = pd.qcut(x[nz], q, duplicates="drop").astype(str)
    return (pd.DataFrame({"x": x, "y": y, "k": k}).groupby("k")
              .agg(x_mid=("x", "median"), n=("y", "size"), mean=("y", "mean"), std=("y", "std")).sort_values("x_mid"))


def add_flags(d):
    d["station_id"] = d["station_id"].astype(str)
    d["group"] = np.select([d.station_id.isin(TIDAL), d.station_id.isin(WEAK_TIDAL), d.station_id.isin(OPERATED_24H),
                            d.station_id.isin(STUCK)], ["조석", "약한 조석", "24h 주기", "고정"], "일반")
    d["nontidal"] = ~d.station_id.isin(TIDAL + STUCK)          # 보고서 기준 '비조석' = 조석 15곳과 고정 1곳을 뺀 나머지
    return d

In [ ]:
cols = ["station_id", "station_code", "hour", "event_block", TARGET] + WL + IMERG + SFC + RADAR + ["dwl_1h", "dwl_3h"]
tr = add_flags(pd.read_parquet(CACHE / "train_sorted.parquet", columns=cols))
nt = tr[tr.nontidal]                                   # 비조석
y = tr[TARGET].to_numpy(float)
print(f"train {len(tr):,}행 · 비조석 {len(nt):,}행")

## 1. 타깃은 어떻게 생겼나

타깃 = **앞으로 6시간 안의 최고 수위 − 지금 수위**(m). 물이 계속 빠지기만 하면 음수가 될 수도 있어요.

In [ ]:
from scipy import stats
print(f"평균 {y.mean():.3f} · 중앙값 {np.median(y):.3f} · 표준편차 {y.std():.3f} · 왜도 {stats.skew(y):.1f} · 첨도 {stats.kurtosis(y):.0f}")
print(f"양수 {(y > 0).mean():.0%} · 0 {(y == 0).mean():.0%} · 음수 {(y < 0).mean():.0%} · 0 또는 ±1cm {(np.abs(y) <= .0101).mean():.0%}")
print(f"p99 {np.quantile(y, .99):.2f} m · 최대 {y.max():.2f} m · 최소 {y.min():.2f} m")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
e = np.logspace(np.log10(.015), 1, 20); bins = np.r_[-e[::-1], [-.005, .005], e]      # 1cm 계단보다 넓은 구간
ax[0].hist(y, bins=bins, color=C_NT); symlog_x(ax[0], .01); ax[0].set_yscale("log")
ax[0].set_xlabel("타깃 (m, symlog)"); ax[0].set_ylabel("행 수 (log)"); ax[0].set_title("0 근처에 몰리고 양쪽 꼬리가 길다")
s = np.sort(np.random.default_rng(0).choice(y, 20000, replace=False))
q = stats.norm.ppf((np.arange(1, len(s) + 1) - .5) / len(s))
ax[1].plot(q, s, ".", ms=2, color=C_NT); ax[1].plot(q, y.mean() + y.std() * q, "--", color=C_ALL, label="정규분포라면")
ax[1].set_xlabel("정규분포의 분위수"); ax[1].set_ylabel("타깃의 분위수 (m)"); ax[1].legend(); ax[1].set_title("Q-Q: 오른쪽 위가 크게 휘어 올라간다")
fig.tight_layout(); plt.show()

**읽는 법 (Q-Q 그림)**: 타깃이 정규분포라면 점이 점선 위에 있어야 해요. 오른쪽 끝에서 점이 점선보다 **훨씬 위로** 올라가요. "정규분포라면 거의 안 나올 큰 값이 꽤 자주 나온다"는 뜻이에요.

### 1-1. 점수는 누가 결정하나 — 제곱 때문에 꼬리가 전부
RMSE는 오차를 **제곱**해요. 1m 틀리면 1, 10cm 틀리면 0.01이에요. **1m 한 번이 10cm 백 번과 같아요.**
그래서 "모든 예측을 0으로 찍었을 때"의 제곱오차가 어디서 나오는지 보면, 점수를 좌우하는 행을 알 수 있어요.

In [ ]:
e2 = np.sort(y ** 2)[::-1]
cum = np.cumsum(e2) / e2.sum()
share = np.arange(1, len(e2) + 1) / len(e2)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(share * 100, cum * 100, color=C_NT); ax.plot([0, 100], [0, 100], ":", color=C_ALL, label="모두 똑같이 기여한다면")
for p in (.01, .05):
    v = cum[int(len(cum) * p) - 1]; ax.scatter(p * 100, v * 100, color=C_TIDAL, zorder=3)
    ax.annotate(f"상위 {p:.0%} 행 → {v:.0%}", (p * 100, v * 100), xytext=(8, -12), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("타깃이 큰 순서로 본 행 비율 (%, log)"); ax.set_ylabel("0 예측 제곱오차 누적 (%)")
ax.legend(); ax.set_title("행 1%가 오차의 2/3"); plt.show()

**상위 1% 행이 제곱오차의 66%** 예요. 그럼 이게 "몇몇 관측소에서만" 생기는 일일까요? (별책에서 본 조석 관측소처럼요.)
**관측소마다** "자기 행 중 상위 1%가 자기 제곱오차의 몇 %인가"를 계산해 봐요. 비조석 관측소 전부를 대상으로 해요.

In [ ]:
def top1_share(v):
    v = np.sort(v.to_numpy(float) ** 2)[::-1]
    k = max(1, len(v) // 100)
    return v[:k].sum() / v.sum() if v.sum() > 0 else np.nan

st_top = nt.groupby("station_id")[TARGET].apply(top1_share)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(st_top * 100, bins=40, color=C_NT); ax.axvline(1, color=C_ALL, ls=":"); ax.text(2, ax.get_ylim()[1] * .85, "꼬리가 없다면 1%", color=C_ALL)
ax.set_xlabel("관측소 자기 행 상위 1%가 차지하는 제곱오차 비중 (%)"); ax.set_ylabel("관측소 수")
ax.set_title(f"비조석 {st_top.notna().sum()}곳 전부: 긴 꼬리는 어디에나 있다"); plt.show()
print(f"관측소별 상위 1% 비중: 10% 지점 {st_top.quantile(.1):.0%} · 중앙값 {st_top.median():.0%} · 90% 지점 {st_top.quantile(.9):.0%}")

관측소 **한 곳 한 곳 안에서도** 상위 1% 행이 제곱오차의 상당 부분(중앙값 48%, 10% 지점도 30%)을 차지해요. 긴 꼬리는 특정 관측소의 문제가 아니라 **모든 하천의 공통 성질**이에요. 평소엔 거의 안 움직이다가 큰 비에 확 불어나니까요.

→ **이 과제는 "큰 상승을 얼마나 잘 맞히나"의 싸움**이에요. 작은 값의 정밀도보다 꼬리가 중요해요(극값 가중치, 손실 함수 실험 후보).

## 2. 강우 세 가지는 어떻게 생겼나

### 2-1. 0이 얼마나 많나

In [ ]:
z = pd.DataFrame({src: {w: (tr[f"{src}_rain_{w}"] == 0).mean() for w in ("1h", "6h", "24h")} for src in ("imerg", "sfc")})
print("강우가 정확히 0인 행 비율"); print((z * 100).round(0).astype(int).astype(str) + "%")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for a, w in zip(ax, ["1h", "6h", "24h"]):
    for src, col, lab in [("imerg", C_NT, "IMERG"), ("sfc", C_RAIN, "지상")]:
        x = tr[f"{src}_rain_{w}"]
        a.hist(np.log10(x[x > 0]), bins=60, histtype="step", lw=1.4, color=col, label=f"{lab} (0: {(x == 0).mean():.0%})")
    a.set_title(f"{w} 누적"); a.set_xlabel("log10 강우 (0 초과만)"); a.legend(fontsize=8)
ax[0].set_ylabel("행 수"); fig.suptitle("양수 부분은 로그 척도에서 넓게 퍼진다", x=.02, ha="left"); fig.tight_layout(); plt.show()

- 지상강수는 1시간 기준 **64%가 0**, IMERG는 33%예요. 지상은 0.5mm 계단이라 약한 비를 0으로 찍어요(1권).
- 양수 부분은 0.1mm부터 100mm 넘게까지 **로그 척도로 넓게** 퍼져요. → 선형 모델이면 `log1p` 변환, 트리 모델이면 그대로 써도 돼요.

### 2-2. 위성과 지상은 같은 비를 재나? (관측소 전부)
같은 곳의 같은 시각이면 두 강우가 비슷해야 해요. 관측소마다 둘의 **순위 상관**을 구해요.

In [ ]:
def st_spearman(d, a, c):
    ra = d[a].groupby(d.station_id).rank(pct=True).to_numpy()
    rc = d[c].groupby(d.station_id).rank(pct=True).to_numpy()
    return station_corr(d, ra, rc)

agree = pd.DataFrame({w: st_spearman(tr, f"imerg_rain_{w}", f"sfc_rain_{w}") for w in ("1h", "6h", "24h")})
fig, ax = plt.subplots(figsize=(9, 3.2))
for w, col in zip(agree.columns, [C_PINK, C_GREEN, C_NT]):
    ax.hist(agree[w].dropna(), bins=50, histtype="step", lw=1.6, color=col, label=f"{w} (중앙값 {agree[w].median():.2f})")
ax.set_xlabel("관측소별 IMERG–지상 순위 상관"); ax.set_ylabel("관측소 수"); ax.legend()
ax.set_title("465곳 전부: 짧은 창은 어긋나고, 길게 모으면 맞아간다"); plt.show()
print("관측소별 순위 상관 (10% · 중앙값 · 90% 지점):"); print(agree.quantile([.1, .5, .9]).round(2))

1시간 단위로는 둘이 꽤 어긋나지만(중앙값 약 0.5), 24시간으로 모으면 잘 맞아요(약 0.75). 거의 모든 관측소에서 같은 패턴이에요.
**예시**: 소나기가 14:50에 왔다면 한쪽은 14시, 다른 쪽은 15시에 기록할 수 있어요. 격자 크기도 달라요. 이런 **시각·위치 차이**는 오래 모으면 상쇄돼요.
→ 세 강우 원천은 **같은 비를 서로 다르게 잰 것**이에요. 평균·최댓값·차이를 묶어 쓰는 방법을 생각할 수 있어요. (두 원천의 시차는 3권에서 수치만 봐요.)

### 2-3. 레이더 세 변수는 사실상 하나

In [ ]:
f = tr.radar_reflectivity_ge20_fraction
print(f"20 이상 화소 비율이 정확히 0 또는 1인 행: {((f == 0) | (f == 1)).mean():.0%}")
smp = tr[RADAR].sample(200_000, random_state=0)
print("레이더 세 변수의 순위 상관:"); print(smp.corr("spearman").round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].hist(tr.radar_reflectivity_mean, bins=100, histtype="step", lw=1.4, color=C_NT, label="평균")
ax[0].hist(tr.radar_reflectivity_p95, bins=100, histtype="step", lw=1.4, color=C_TIDAL, label="p95")
ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_xlabel("반사도 (단위 미기재)"); ax[0].set_title("평균·p95")
m = smp.radar_reflectivity_mean
ax[1].scatter(m, smp.radar_reflectivity_ge20_fraction, s=1, alpha=.1, color=C_NT)
ax[1].set_xlabel("반사도 평균"); ax[1].set_ylabel("20 이상 화소 비율"); ax[1].set_title("평균이 약 20을 넘으면 비율이 0 → 1로 뛴다")
fig.tight_layout(); plt.show()

"20 이상 화소 비율"은 **평균 반사도에 문턱(20)을 건 것**과 거의 같아요. 평균이 14 아래면 0, 24 위면 1이에요. 세 변수의 상관도 0.9 이상이라, **레이더는 사실상 변수 하나**예요.

## 3. 비가 오면 타깃은 어떻게 변하나

강우량을 20칸(0은 따로 한 칸)으로 나누고, 칸마다 **타깃 평균(실선)** 과 **표준편차(점선)** 를 그려요. 전체(회색)와 비조석(파랑)을 나란히 놓아요. 조석 관측소는 비와 상관없이 움직여서 관계를 흐리거든요(별책).

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
res = {}
for a, c in zip(ax.ravel(), IMERG + SFC):
    for g, d, col in [("전체", tr, C_ALL), ("비조석", nt, C_NT)]:
        bb = binned(d[c], d[TARGET])
        a.plot(bb.x_mid, bb["mean"], "-o", ms=2.5, color=col, label=f"{g} 평균")
        a.plot(bb.x_mid, bb["std"], ":", color=col, label=f"{g} 표준편차")
        res[(c, g)] = {"비 0일 때 평균": bb["mean"].iloc[0], "최다 칸 평균": bb["mean"].iloc[-1],
                       "비 0일 때 표준편차": bb["std"].iloc[0], "최다 칸 표준편차": bb["std"].iloc[-1]}
    symlog_x(a, .1); a.set_title(c, fontsize=9)
ax[0, 0].legend(fontsize=7); ax[0, 0].set_ylabel("타깃 (m)"); ax[1, 0].set_ylabel("타깃 (m)")
for a in ax[1]: a.set_xlabel("강우 (mm, symlog)")
fig.suptitle("비가 많을수록 평균도, 흩어짐도 커진다", x=.02, ha="left"); fig.tight_layout(); plt.show()
pd.DataFrame(res).T.round(3)

**뭐가 보이나요?**
1. **약 1mm를 넘으면 평균이 가파르게 올라요.** 비조석 기준 지상 1h: 비 0일 때 0.04m → 가장 많은 칸 0.65m.
2. **표준편차(점선)도 같이 커져요(이분산).** 비가 많을 때는 "크게 오를 수도, 별로 안 오를 수도" 있어요.
   - **예시**: 30mm 비가 와도 이미 물이 빠지는 중인 대하천은 조금 오르고, 소하천은 몇 m 오를 수 있어요.
3. 비가 **없을 때**의 흩어짐은 전체보다 비조석에서 훨씬 작아요. 비 없이 흔들리는 건 대부분 조석이에요(별책).
4. **짧은 창(1h·6h)과 지상강수** 곡선이 가장 가팔라요.

→ 큰 비일수록 예측이 본질적으로 어려워요. 평균만 맞히는 손실(MSE)로도 되지만, **큰 비 구간에 가중치를 주거나 분위수를 같이 예측**하는 실험을 해 볼 만해요.

### 3-1. 레이더도 같은 모양

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for a, c in zip(ax, RADAR):
    for g, d, col in [("전체", tr, C_ALL), ("비조석", nt, C_NT)]:
        bb = binned(d[c], d[TARGET])
        a.plot(bb.x_mid, bb["mean"], "-o", ms=2.5, color=col, label=g)
    a.set_title(c.replace("radar_reflectivity_", "레이더 "), fontsize=9); a.set_xlabel("값")
ax[0].set_ylabel("타깃 평균 (m)"); ax[0].legend(); fig.tight_layout(); plt.show()

반사도가 높을수록 타깃이 커요(비조석 평균·p95 최상위 칸 약 0.45m). 강우와 비슷한 크기의 신호예요.

## 4. 이 관계가 관측소 전부에서 성립하나?

3장은 모든 관측소를 섞은 그림이에요. "몇몇 큰 관측소가 만든 관계"일 수도 있으니, **관측소 한 곳씩** 확인해요.
1. **방향**: 관측소마다 "6시간 강우(두 원천 중 큰 값)의 순위 ↔ 타깃의 순위" 상관
2. **이분산**: 관측소마다 "비 많은 행(상위 10%)의 타깃 표준편차 ÷ 비 없는 행의 타깃 표준편차"

In [ ]:
d = tr.copy()
d["rain6"] = np.maximum(d.imerg_rain_6h, d.sfc_rain_6h)
rho = station_corr(d, d.rain6.groupby(d.station_id).rank(pct=True).to_numpy(), d[TARGET].groupby(d.station_id).rank(pct=True).to_numpy())

def hetero(g):
    wet = g[g.rain6 >= g.rain6.quantile(.9)][TARGET].std()
    dry = g[g.rain6 == 0][TARGET].std()
    return wet / dry if (dry and dry > 0 and (g.rain6 == 0).sum() > 30) else np.nan
ratio = d.groupby("station_id").apply(hetero)

grp = d.groupby("station_id").group.first()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for k, col in [("일반", C_NT), ("조석", C_TIDAL)]:
    m = grp == k
    ax[0].hist(rho[m].dropna(), bins=np.linspace(-.2, .8, 51), color=col, alpha=.7, label=f"{k} ({m.sum()}곳)")
    ax[1].hist(np.log2(ratio[m].dropna()), bins=40, color=col, alpha=.7, label=k)
ax[0].axvline(0, color="k", lw=.6); ax[0].set_xlabel("관측소별 강우–타깃 순위 상관"); ax[0].set_ylabel("관측소 수"); ax[0].legend()
ax[0].set_title("① 방향: 일반 관측소는 거의 전부 +")
ax[1].axvline(0, color="k", lw=.6); ax[1].set_xlabel("log2(비 많을 때 표준편차 ÷ 비 없을 때)  · 0 = 같음, 1 = 2배")
ax[1].set_title("② 이분산: 거의 전부 비 올 때 더 들쭉날쭉"); fig.tight_layout(); plt.show()
gen = grp == "일반"
print(f"일반 관측소 {gen.sum()}곳: 상관 > 0 인 곳 {(rho[gen] > 0).mean():.1%} · 중앙값 {rho[gen].median():.2f}")
print(f"일반 관측소: 비 많을 때 표준편차가 더 큰 곳 {(ratio[gen] > 1).mean():.1%} · 배율 중앙값 {ratio[gen].median():.1f}배")

- ① 일반 관측소는 **거의 전부** 비와 타깃이 같은 방향으로 움직여요. 조석 관측소만 0 근처에 모여 있어요.
- ② 일반 관측소의 **99%** 가 비가 많을 때 타깃이 더 들쭉날쭉해요. 표준편차 배율의 중앙값은 **약 10배**예요.

→ "비가 오면 오르고, 많이 올수록 불확실하다"는 건 **몇 곳의 이야기가 아니라 일반 하천 전체의 규칙**이에요. 반대로 **얼마나 세게 반응하나**(상관의 크기)는 관측소마다 꽤 달라요. 이 차이는 4권(관측소 유형)에서 다뤄요.

## 5. 변수별 상관 요약 — Pearson vs Spearman

두 상관을 나란히 놓으면 관계의 **모양**을 짐작할 수 있어요.
- 둘 다 비슷 → 대체로 직선 관계
- **Spearman ≫ Pearson** → "계속 같은 방향이긴 한데 곧은 직선은 아님"(곡선이나 극값 영향). 트리 모델이 유리해요.

비조석에서 30만 행을 뽑아 계산해요.

In [ ]:
smp = nt.sample(300_000, random_state=0)
feats = ["wl_t", "dwl_1h", "dwl_3h"] + IMERG + SFC + RADAR
t = pd.DataFrame({f: {"Pearson": smp[f].corr(smp[TARGET]), "Spearman": smp[f].corr(smp[TARGET], method="spearman")} for f in feats}).T
t = t.sort_values("Spearman")
fig, ax = plt.subplots(figsize=(9, 5))
yy = np.arange(len(t))
ax.barh(yy + .2, t.Spearman, .4, color=C_NT, label="Spearman (순서)"); ax.barh(yy - .2, t.Pearson, .4, color=C_RAIN, label="Pearson (직선)")
ax.set_yticks(yy); ax.set_yticklabels(t.index, fontsize=8); ax.axvline(0, color="k", lw=.5); ax.legend(loc="lower right")
ax.set_xlabel("타깃과의 상관 (비조석)"); ax.set_title("Spearman이 Pearson보다 일관되게 크다: 단조롭지만 비선형"); plt.show()
t["Pearson÷Spearman"] = t.Pearson / t.Spearman
t.round(3)

- 가장 강한 변수는 **최근 수위 변화(`dwl_1h`, `dwl_3h`)** 와 **짧은 창 강우(지상 1h·6h)**, 레이더예요(Spearman 0.35~0.41).
- 강우·레이더·수위 변화 모두 **Pearson이 Spearman보다 작아요**(대략 0.7~0.95배). 순서로는 같이 가는데 곧은 선은 아니라는 뜻이에요(1mm 넘으면 가팔라지는 모양, 3장).
- 24h 누적 강우만 둘이 비슷하거나 반대예요. 긴 창 강우는 관계 자체가 약해요(0.26).
- 절대 수위 `wl_t`는 거의 0이에요. 관측소마다 기준면이 달라서 절대값은 의미가 없어요(1권).

→ 비선형을 알아서 잡는 **트리 계열 모델**이 잘 맞아요. 선형 모델을 쓴다면 `log1p` 변환과 교차항이 필요해요.

## 6. 누수 점검과 정리

### 누수 점검표

| 어디 | 미래 정보? | 판단 |
|---|---|---|
| 1장 타깃 분포·꼬리 | 타깃 자체(결과 쪽) | 설명용. 모델 입력 아님 |
| 2장 강우·레이더 분포, 원천 간 상관 | 입력값(모두 '지금까지' 누적, 데이터 사전 기준) | 누수 없음 |
| 3·4장 강우 × 타깃 | 입력은 과거 누적 강우, 타깃은 결과 쪽 | 관계 설명용. 누수 없음 |
| 4장 관측소별 반응 크기 | 타깃을 씀 | 모델에 **관측소 특성으로 넣을 땐 학습 묶음 안에서만** 계산 |
| 5장 상관 요약 | 동일 시점 입력 ↔ 타깃 | 설명용 |

### 정리

| 발견 | 전체에 해당? | 모델링 시사점 |
|---|---|---|
| 상위 1% 행이 0 예측 제곱오차의 66% | ✅ 비조석 관측소 각각에서도 꼬리가 큼 | 꼬리 예측에 집중: 극값 가중치·손실 함수 실험 |
| 강우 0이 많고(지상 1h 64%), 양수는 로그로 넓게 퍼짐 | 전 행 | `log1p`(선형 모델), 트리는 그대로 |
| 위성–지상 일치는 1h 약 0.5 → 24h 약 0.75 | ✅ 관측소 거의 전부 | 원천을 평균·최대·차이로 묶기 |
| 레이더 세 변수는 사실상 하나 | 전 행 | 하나(평균 또는 p95)로 축약 가능 |
| 비가 많을수록 타깃 평균·분산 모두 증가 | ✅ 일반 관측소 거의 전부 | 큰 비 구간 가중치·분위수 예측 검토 |
| 반응 크기는 관측소마다 다름 | 관측소별 분포 | 관측소 특성(4권), 학습 묶음 안에서 계산 |
| Spearman > Pearson (0.7~0.95배) | 짧은 창 변수 전반 | 트리 계열 모델 |